# Source B — Home Systems (Reality scenario)

Some off-grid households already own their own electricity equipment: a solar panel, a small generator ("motor propio"), or something else — this is recorded in the 2024 census. This notebook turns those counts into per-cluster installed capacities for EnergyScope (`PV_HS`, `HS_DIESEL`, `BATT_HS`), and works out how much of each cluster's electricity comes from off-grid households (Source B) versus the total.

Output: `output_energyscope/source_B_home_systems_reality.csv`, one row per cluster.

In [35]:
import pandas as pd

# Municipalities grouped into the 5 clusters used throughout this project (same as demande.ipynb)
CLUSTERS = {
    1: ["Exaltación", "Reyes", "Santa_Rosa_Beni", "Ixiamas"],
    2: ["Bolpebra"],
    3: ["Guayaramerín", "Riberalta", "Puerto_Gonzalo_Moreno"],
    4: ["Bella_Flor", "Filadelfia", "Ingavi", "Nueva_Esperanza", "Porvenir",
        "Puerto_Rico", "San_Lorenzo", "San_Pedro", "Santa_Rosa_Pando",
        "Santos_Mercado", "Sena", "Villa_Nueva"],
    5: ["Cobija"],
}
MUNI_TO_CLUSTER = {muni: k for k, munis in CLUSTERS.items() for muni in munis}

## Assumptions

The three values below aren't in any source file — they're rough sizing assumptions needed to turn household *counts* into installed *capacity* (W, kWh).

In [36]:
# Source: ENDE BO-L1222 / Programa de Electrificación Rural
PANEL_W = 50       # Wp per panel-HH (ENDE PEVD kit: ≥50 Wp policristalino)
GEN_W = 500        # W per generator-HH (assumption, no field data — sensitivity at 350W)
BATT_KWH_PER_PANEL_HH = 0.123  # kWh per panel-HH (ENDE PEVD kit: 123 Wh lithium 12V)

## 1. Home-system equipment from the census

`CSV_final.csv` (2024 census) records, per municipality, how many households get their electricity from their own solar panel, their own generator ("motor propio"), or another source ("otra"). Municipality names are matched to RAMP's naming using the same `CSVFINAL_TO_RAMP` table as in `demande.ipynb` (two "Santa Rosa" require the department to disambiguate).

In [37]:
# Census file has two "Santa Rosa" — disambiguated by (name, department)
CSVFINAL_TO_RAMP = {
    ("Ixiamas",               "La Paz"): "Ixiamas",
    ("Riberalta",             "Beni"):   "Riberalta",
    ("Guayaramerín",          "Beni"):   "Guayaramerín",
    ("Reyes",                 "Beni"):   "Reyes",
    ("Santa Rosa",            "Beni"):   "Santa_Rosa_Beni",
    ("Exaltación",            "Beni"):   "Exaltación",
    ("Cobija",                "Pando"):  "Cobija",
    ("Porvenir",              "Pando"):  "Porvenir",
    ("Bolpebra",              "Pando"):  "Bolpebra",
    ("Bella Flor",            "Pando"):  "Bella_Flor",
    ("Puerto Rico",           "Pando"):  "Puerto_Rico",
    ("San Pedro",             "Pando"):  "San_Pedro",
    ("Filadelfia",            "Pando"):  "Filadelfia",
    ("Puerto Gonzalo Moreno", "Pando"):  "Puerto_Gonzalo_Moreno",
    ("San Lorenzo",           "Pando"):  "San_Lorenzo",
    ("Sena",                  "Pando"):  "Sena",
    ("Santa Rosa",            "Pando"):  "Santa_Rosa_Pando",
    ("Ingavi",                "Pando"):  "Ingavi",
    ("Nueva Esperanza",       "Pando"):  "Nueva_Esperanza",
    ("Villa Nueva",           "Pando"):  "Villa_Nueva",
    ("Santos Mercado",        "Pando"):  "Santos_Mercado",
}

# Columns in CSV_final.csv (0-indexed, no header row kept so positions stay fixed):
# 1 = department, 3 = municipality, 16 = Motor propio 2024, 17 = Panel solar 2024, 18 = Otra 2024
census = pd.read_csv("../../exctraction of data/output/CSV_final.csv", header=None)

def to_int(x):
    s = str(x).strip().replace(" ", "").replace("\xa0", "")
    return int(float(s)) if s not in ("", "-", "nan") else 0

rows = []
for _, row in census.iloc[1:].iterrows():
    key = (str(row[3]).strip(), str(row[1]).strip())
    if key not in CSVFINAL_TO_RAMP:
        continue
    rows.append({
        "municipality": CSVFINAL_TO_RAMP[key],
        "hh_panels": to_int(row[17]),
        "hh_generators": to_int(row[16]),
        "hh_other": to_int(row[18]),
    })

hh = pd.DataFrame(rows)
hh["cluster"] = hh["municipality"].map(MUNI_TO_CLUSTER)

print("Per-municipality equipment counts (Source: CSV_final.csv, 2024 census):")
print(hh.to_string(index=False))

by_cluster = hh.groupby("cluster")[["hh_panels", "hh_generators", "hh_other"]].sum()

Per-municipality equipment counts (Source: CSV_final.csv, 2024 census):
         municipality  hh_panels  hh_generators  hh_other  cluster
              Ixiamas        353            337       103        1
            Riberalta        692            422       282        3
         Guayaramerín        383            166        97        3
                Reyes        213             96        44        1
      Santa_Rosa_Beni        405            198        17        1
           Exaltación        664            203        23        1
               Cobija        153             50        45        5
             Porvenir        121             61        35        4
             Bolpebra        216            160        17        2
           Bella_Flor        160             67        88        4
          Puerto_Rico         67            233        20        4
            San_Pedro         84            136        21        4
           Filadelfia        240            334        34

## 2. Splitting "Otra" between PV and diesel

The census doesn't say what kind of system the "otra" households have, so we just split them evenly in two: half goes to PV, half goes to diesel.

In [38]:
by_cluster["hh_other_to_pv"] = by_cluster["hh_other"] / 2
by_cluster["hh_other_to_diesel"] = by_cluster["hh_other"] / 2

by_cluster[["hh_other", "hh_other_to_pv", "hh_other_to_diesel"]]

,hh_other,hh_other_to_pv,hh_other_to_diesel
cluster,,,
1,187,93.5,93.5
2,17,8.5,8.5
3,505,252.5,252.5
4,498,249.0,249.0
5,45,22.5,22.5


## 3. Installed capacities

Household counts × unit size = installed capacity. This is existing equipment, not something to be built, so `f_max = f_min` for all three technologies.

In [39]:
# GW = W * 1e-9, GWh = kWh * 1e-6
by_cluster["f_min_PV_HS_GW"] = (by_cluster["hh_panels"] + by_cluster["hh_other_to_pv"]) * PANEL_W * 1e-9
by_cluster["f_min_HS_DIESEL_GW"] = (by_cluster["hh_generators"] + by_cluster["hh_other_to_diesel"]) * GEN_W * 1e-9
by_cluster["f_min_BATT_HS_GWh"] = (by_cluster["hh_panels"] + by_cluster["hh_other_to_pv"]) * BATT_KWH_PER_PANEL_HH * 1e-6

by_cluster[["f_min_PV_HS_GW", "f_min_HS_DIESEL_GW", "f_min_BATT_HS_GWh"]]

,f_min_PV_HS_GW,f_min_HS_DIESEL_GW,f_min_BATT_HS_GWh
cluster,,,
1,0.000086,0.000464,0.000213
2,0.000011,0.000084,0.000028
3,0.000069,0.000444,0.000170
4,0.000071,0.001168,0.000175
5,0.000009,0.000036,0.000022


## 4. Electricity demand: Source A vs Source B

Source A is grid electricity sold, measured by AETN. Source B is the off-grid RAMP simulation. Both files already give electrical energy in GWh/MWh, per municipality. Cooking is excluded from both — it's not electric and is handled separately in `demande.ipynb`.

In [40]:
# Source A: grid electricity sold (AETN), already split by municipality/sector/end_use
source_a = pd.read_csv("../../exctraction of data/output/source_A_all_sectors_end_uses.csv")
source_a["cluster"] = source_a["muni_ramp"].map(MUNI_TO_CLUSTER)
source_a_gwh = source_a[source_a["end_use"] != "COOKING"].groupby("cluster")["MWh"].sum() / 1000

# Source B: off-grid RAMP simulation, one row per municipality, one column per appliance.
# "restaurant_kitchen" is the only appliance that maps to cooking, so it's the only one dropped.
ramp = pd.read_csv("data ramp/ramp_reality_annual_summary.csv")
ramp = ramp[ramp["municipality"] != "TOTAL"].copy()
ramp["cluster"] = ramp["municipality"].map(MUNI_TO_CLUSTER)
source_b_gwh = ramp.groupby("cluster")["TOTAL_GWh"].sum() - ramp.groupby("cluster")["restaurant_kitchen"].sum()

by_cluster["sourceB_elec_demand_GWh"] = source_b_gwh
by_cluster["total_elec_demand_GWh"] = source_a_gwh + source_b_gwh

print("share_dispersion = sourceB_elec_demand_GWh / total_elec_demand_GWh")
by_cluster["share_dispersion"] = by_cluster["sourceB_elec_demand_GWh"] / by_cluster["total_elec_demand_GWh"]

by_cluster[["sourceB_elec_demand_GWh", "total_elec_demand_GWh", "share_dispersion"]].round(4)

share_dispersion = sourceB_elec_demand_GWh / total_elec_demand_GWh


,sourceB_elec_demand_GWh,total_elec_demand_GWh,share_dispersion
cluster,,,
1,0.7911,12.4370,0.0636
2,0.1032,0.3425,0.3014
3,0.7942,89.7794,0.0088
4,1.1465,23.7252,0.0483
5,0.1220,54.4706,0.0022


## 5. Save

In [43]:
out = by_cluster.reset_index()[[
    "cluster", "f_min_PV_HS_GW", "f_min_HS_DIESEL_GW", "f_min_BATT_HS_GWh",
    "sourceB_elec_demand_GWh", "total_elec_demand_GWh", "share_dispersion",
]]
out_path = "output_energyscope/source_B_home_systems_reality.csv"
out.to_csv(out_path, index=False)
print(f"Saved {out_path}")
out

Saved output_energyscope/source_B_home_systems_reality.csv


,cluster,f_min_PV_HS_GW,f_min_HS_DIESEL_GW,f_min_BATT_HS_GWh,sourceB_elec_demand_GWh,total_elec_demand_GWh,share_dispersion
0,1,0.000086,0.000464,0.000213,0.791138,12.437040,0.063611
1,2,0.000011,0.000084,0.000028,0.103235,0.342484,0.301428
2,3,0.000069,0.000444,0.000170,0.794245,89.779421,0.008847
3,4,0.000071,0.001168,0.000175,1.146498,23.725166,0.048324
4,5,0.000009,0.000036,0.000022,0.121981,54.470599,0.002239


## 6. Which appliances drive Source B demand?

Same RAMP simulation, broken down by appliance instead of summed together — just to see what to target first if Source B demand looks too high. Not saved to file, only shown here.

In [42]:
appliance_cols = [c for c in ramp.columns if c not in ("municipality", "TOTAL_GWh", "cluster")]

by_appliance = ramp.groupby("cluster")[appliance_cols].sum().T
by_appliance["TOTAL"] = by_appliance.sum(axis=1)
by_appliance = by_appliance.sort_values("TOTAL", ascending=False)
by_appliance.columns = [f"C{c}" if c != "TOTAL" else "TOTAL" for c in by_appliance.columns]

by_appliance.round(3)

,C1,C2,C3,C4,C5,TOTAL
sufficiency_cold_storage,0.166,0.014,0.201,0.232,0.049,0.662
sufficiency_illumination,0.148,0.022,0.126,0.210,0.014,0.520
workshop_machinery,0.126,0.018,0.108,0.174,0.012,0.437
sufficiency_thermal_comfort,0.071,0.010,0.105,0.137,0.017,0.340
store_cold_storage,0.081,0.012,0.070,0.112,0.008,0.283
restaurant_cold_storage,0.061,0.009,0.053,0.085,0.006,0.214
sufficiency_ICT,0.048,0.006,0.052,0.072,0.009,0.187
rice_processing_rice_processing,0.041,0.004,0.037,0.056,0.004,0.141
entertainment_business_cold_storage,0.040,0.006,0.034,0.055,0.004,0.138
store_illumination,0.004,0.001,0.004,0.006,0.000,0.014
